# **07. Exception Object and Error Information**

### **7.1 Understanding Exception Objects**

**1. What is an Exception Object?**

- ***When an exception occurs, Python creates an exception object.***

- ***This object often contains useful information about the error.***

- ***Every exception object is an instance of an exception class (for example, `ValueError`).***

In [ ]:
try:
    int("abc")
except ValueError as e:
    print(type(e))                    # the class of the object
    print(e.args)                     # arguments stored in the object
    print(str(e))                     # readable message
    print(repr(e))                    # developer-friendly form

### **7.2 Capturing Exception Messages Using `as`**

**1. Using `as e`**

- ***You can capture the exception object using `as e` (or any other variable name).***

- ***Printing `e` shows the error message.***

In [ ]:
data = [1, 2, 3]

for index in ["1", "5", "x"]:
    try:
        value = data[int(index)]
        print(f"Value at index {index}: {value}")
    except IndexError as e:                        # 'e' will hold the IndexError object
        print(f"Error accessing list: {e}")
    except ValueError as e:                        # 'e' will hold the ValueError object
        print(f"Error converting input: {e}")

**2. Scope of `e`**

- ***The variable `e` exists only inside the `except` block. Python deletes it after the block ends.***

- ***If you need the error later, store it in another variable.***

In [ ]:
try:
    10 / 0
except ZeroDivisionError as e:
    saved_error = e                                # keep a copy

try:
    print(e)
except NameError as err:
    print("e is not available here:", err)

print("saved_error:", saved_error)

### **7.3 Exception Class Hierarchy**

**1. Exception Hierarchy**

- ***All exceptions are classes arranged in a hierarchy.***

- ***A handler for a parent class also catches its child classes.***

- ***`BaseException` is at the top. Most errors inherit from `Exception`.***

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 └── Exception
      ├── ArithmeticError
      │    ├── ZeroDivisionError
      │    └── OverflowError
      ├── AssertionError
      ├── AttributeError
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── NameError
      ├── OSError
      │    ├── FileNotFoundError
      │    ├── FileExistsError
      │    ├── PermissionError
      │    └── IsADirectoryError
      ├── RuntimeError
      │    └── RecursionError
      ├── TypeError
      └── ValueError
           └── UnicodeError
                └── UnicodeDecodeError
```

In [ ]:
for exc in [ZeroDivisionError, KeyError, FileNotFoundError, ModuleNotFoundError]:
    parents = [c.__name__ for c in exc.__mro__[1:-1]]
    print(exc.__name__, "->", parents)

**2. Catching a Parent Class**

- ***`ArithmeticError` catches `ZeroDivisionError` and `OverflowError`.***

- ***`LookupError` catches `IndexError` and `KeyError`.***

In [ ]:
try:
    10 / 0
except ArithmeticError as e:
    print("ArithmeticError caught:", type(e).__name__)

try:
    {"a": 1}["b"]
except LookupError as e:
    print("LookupError caught:", type(e).__name__)

**3. Why a Bare `except:` is Dangerous**

- ***`KeyboardInterrupt` and `SystemExit` inherit from `BaseException`, not from `Exception`.***

- ***`except Exception` does not catch them, but a bare `except:` does, which hides keyboard interrupts.***

### **7.4 Understanding Tracebacks**

**1. What is a Traceback?**

- ***When an exception is not handled, Python prints a traceback.***

- ***A traceback shows the path the program took to reach the error.***

```
>>> set_age(200)       # This will raise a ValueError
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
  File "<stdin>", line 6, in set_age
ValueError: Age must be between 0 and 120.
```

**2. How to Read a Traceback**

1. ***Start from the last line: it shows the exception type and its message.***
2. ***The lines above show the chain of calls, with the most recent call last.***
3. ***Each entry shows the file, the line number and the function name.***
4. ***The first entry is where the program started, and the last entry is where the error occurred.***

In [ ]:
try:
    10 / 0
except ZeroDivisionError as e:
    tb = e.__traceback__
    print("Error occurred on line:", tb.tb_lineno)

### **7.5 Using the `traceback` Module**

**1. What is the `traceback` Module?**

- ***The `traceback` module lets you work with traceback information in your own code.***

- ***`traceback.format_exc()` returns the full traceback as a string.***

- ***`traceback.print_exc()` prints the traceback.***

- ***`traceback.extract_tb()` gives the individual steps of the traceback.***

In [ ]:
import traceback

def divide(a, b):
    return a / b

try:
    divide(1, 0)
except ZeroDivisionError:
    text = traceback.format_exc()
    print(text)

In [ ]:
try:
    divide(1, 0)
except ZeroDivisionError as e:
    for step in traceback.extract_tb(e.__traceback__):
        print("Function:", step.name, "| Line:", step.lineno)

**2. Logging the Exception**

- ***In production code, log the exception instead of hiding it.***

- ***`logging.exception(...)` writes the message together with the traceback.***

In [ ]:
import logging

logging.basicConfig(level=logging.ERROR)

try:
    divide(1, 0)
except ZeroDivisionError:
    logging.exception("Division failed")